In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from simulate.interpolant import Interpolant


In [2]:
param = (
    pd.read_csv("data/parameters.csv")
    .drop_duplicates()
    .sort_values(["Mode", "State of Charge / 1"])
    .reset_index(drop=True)
)
keys = [
    "Mode",
    "State of Charge / 1",
    "Open-circuit voltage / V",
    "R0 / Ohm",
    "R1 / Ohm",
    "R2 / Ohm",
    "Tau1 / s",
    "Tau2 / s",
]
param[keys]

,Mode,State of Charge / 1,Open-circuit voltage / V,R0 / Ohm,R1 / Ohm,R2 / Ohm,Tau1 / s,Tau2 / s
0,Charge,0.004664,2.937549,0.057024,0.447728,0.294898,297.414337,2911.581090
1,Charge,0.008822,2.998336,0.049115,0.367057,0.939422,327.287084,4257.434217
2,Charge,0.012979,3.046779,0.045776,0.409908,0.954288,410.675689,6389.122493
3,Charge,0.017135,3.087881,0.043726,0.369877,0.924684,418.579867,7415.677275
4,Charge,0.021291,3.123427,0.042308,0.343183,0.967973,444.876786,7693.189594
...,...,...,...,...,...,...,...,...
478,Discharge,0.979238,4.134464,0.020683,0.013602,0.076885,16.816196,682.474870
479,Discharge,0.983394,4.141345,0.020880,0.014181,0.138028,20.850761,1205.664664
480,Discharge,0.987550,4.149105,0.020425,0.013904,0.074684,15.880308,534.188373
481,Discharge,0.991706,4.158130,0.020663,0.014314,0.079377,18.429019,517.266851


In [3]:
kinds = {
    "Open-circuit voltage / V": "lut",
    "R0 / Ohm": "spline",
    "R1 / Ohm": "spline",
    "R2 / Ohm": "spline",
    "Tau1 / s": "spline",
    "Tau2 / s": "spline",
}
options = {
    "Open-circuit voltage / V": {"kind": "cubic"},
    "R0 / Ohm": {"k": 1, "n": 3},
    "R1 / Ohm": {"k": 1, "n": 3},
    "R2 / Ohm": {"k": 1, "n": 3},
    "Tau1 / s": {"k": 1, "n": 3},
    "Tau2 / s": {"k": 1, "n": 3},
}

for key in keys[2:]:
    fig = px.scatter(
        param,
        x="State of Charge / 1",
        y=key,
        color="Mode",
    )
    for mode in param["Mode"].unique():
        mask = param["Mode"] == mode
        x = param.loc[mask, "State of Charge / 1"].values
        y = param.loc[mask, key].values
        interpolant = Interpolant(x, y, kind=kinds[key], options=options[key])
        x_interp = np.linspace(0, 1, 1000)
        y_interp = interpolant(x_interp)
        fig.add_trace(
            go.Scatter(
                x=x_interp,
                y=y_interp,
                mode="lines",
            )
        )
    fig.show()